# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedant-Jagtap/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
from datasets import load_dataset
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

march_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    data_files={"train": "fact_content_daily_performance/month=2026-03/data_0.parquet"},
    token=HF_TOKEN
)

march_df = march_ds.to_pandas()

print(march_df.shape)
march_df.head()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

(9841378, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
### Rule

Pages with low click-through performance and low organic traffic are candidates for content refresh.

### Action Label

REFRESH_CONTENT

### Reason Code

LOW_CTR_LOW_TRAFFIC

### Why

If a page receives impressions but relatively few clicks and low organic traffic, the content may be outdated, poorly optimized, or less relevant to current search intent.

In [3]:
march_df["ctr"] = (
    march_df["gsc_clicks"] /
    march_df["gsc_impressions"].replace(0, np.nan)
)

march_df["ctr_bucket"] = pd.cut(
    march_df["ctr"],
    bins=[-1, 0.01, 0.03, 0.05, 1],
    labels=["Very Low", "Low", "Medium", "High"]
)

ctr_table = (
    march_df
    .groupby("ctr_bucket")
    .agg(
        n=("ctr", "count"),
        avg_clicks=("gsc_clicks", "mean")
    )
)

print(ctr_table)

/tmp/ipykernel_1076/2863854166.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("ctr_bucket")


                  n  avg_clicks
ctr_bucket                     
Very Low    3414297    0.125529
Low          132328    2.231002
Medium        29724    1.743911
High          34712    1.330376


Verdict: CONFIRMED

Pages with very low CTR show weaker engagement and are reasonable candidates for refresh actions.

In [5]:
march_df["traffic_bucket"] = pd.cut(
    march_df["sessions_organic"].fillna(0),
    bins=[-1, 0, 10, 100, march_df["sessions_organic"].max()],
    labels=["Zero", "Low", "Medium", "High"]
)

traffic_table = (
    march_df
    .groupby("traffic_bucket", observed=False)
    .agg(
        n=("sessions_organic", "count"),
        avg_sessions=("sessions_organic", "mean")
    )
)

print(traffic_table)

                      n  avg_sessions
traffic_bucket                       
Zero            6609994      0.000000
Low              206946      2.298015
Medium             5659     18.460329
High                 38    141.473684


Verdict: CONFIRMED

The majority of pages have zero or very low organic traffic. Pages with low traffic are reasonable candidates for content refresh because they are attracting limited organic visibility.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
import numpy as np

# CTR
march_df["ctr"] = (
    march_df["gsc_clicks"] /
    march_df["gsc_impressions"].replace(0, np.nan)
)

march_df["ctr"] = march_df["ctr"].fillna(0)

# Score components
ctr_score = (1 - march_df["ctr"].clip(0, 1)) * 70

traffic_score = (
    1 -
    (
        march_df["sessions_organic"].fillna(0) /
        march_df["sessions_organic"].max()
    )
) * 30

march_df["baseline_score"] = (
    ctr_score + traffic_score
)

march_df["action_label"] = "REFRESH_CONTENT"

march_df["reason_code"] = "LOW_CTR_LOW_TRAFFIC"

queue = (
    march_df
    .sort_values(
        "baseline_score",
        ascending=False
    )
)

os.makedirs(
    "work/outputs",
    exist_ok=True
)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(
    "CSV written successfully"
)

queue[
    [
        "content_hash_id",
        "baseline_score",
        "action_label",
        "reason_code"
    ]
].head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.